In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.9 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
import re
import pandas as pd
from groq import Groq, RateLimitError
from tqdm import tqdm
import math

In [ ]:


# ---------------------------------------------------------
# 1. SETUP GROQ CLIENT
# ---------------------------------------------------------
# ⚠️ CHANGE THIS TO YOUR NEW API KEY AFTER YOU REVOKE THE OLD ONE
GROQ_API_KEY = ""
client = Groq(api_key=GROQ_API_KEY)

MODEL = "openai/gpt-oss-20b"
EXAMPLES_PER_CLASS = 200
BATCH_SIZE = 10

# ---------------------------------------------------------
# 2. MASTER PROMPTS
# ---------------------------------------------------------
class_instructions = {
    "Stress": "Focus on external pressures: work, bosses, exams, money, deadlines. Use colloquial idioms like 'killing me', 'burned out', 'drowning in work', 'freaking out'. Make it sound like a frustrated Reddit post.",
    "Anxiety": "Focus on internal cognitive loops and dread: 'what if', overthinking, fear of the future, physical restlessness, racing heart. Do NOT just talk about external work; focus on the internal panic.",
    "Suicidal": "Strictly about literal ideation, wanting to die, or wanting to cease existing. CRITICAL: DO NOT use idioms like 'this is killing me' or 'I'm dead'. Must be serious, dark, and literal.",
    "Depression": "Focus on apathy, numbness, lack of energy, hopelessness, isolation, heavy sadness, and inability to get out of bed. Less about 'panic' and more about 'emptiness'.",
    "Normal": "Mundane daily activities, hobbies, neutral or positive observations, chores, weather. CRITICAL: NO complaints, NO stress, NO hidden negative emotions. Just a normal person updating their status.",
    "Bipolar": "Focus on manic episodes: grandiosity, no need for sleep, racing thoughts, impulsive spending, rapid speech, feeling invincible or like a god.",
    "Personality disorder": "Focus on Borderline Personality Disorder (BPD) traits: intense fear of abandonment, splitting (idealizing then devaluing people), identity disturbance, intense and unstable relationships."
}

# ---------------------------------------------------------
# 3. LOAD EXISTING DATA & PRINT STATUS DASHBOARD
# ---------------------------------------------------------
all_data = []
checkpoint_file = "/content/drive/MyDrive/Datasets/HackforHumanity/checkpoint_mental_health.csv"

# Optional: Load previous checkpoint if the script crashed earlier
if os.path.exists(checkpoint_file):
    print(f"♻️ Found existing {checkpoint_file}. Resuming from previous progress...")
    df_resume = pd.read_csv(checkpoint_file)
    all_data = df_resume.to_dict('records')
    print(f"Loaded {len(all_data)} previously generated rows.")

# Count current rows per class to build the dashboard
class_counts = {cat: 0 for cat in class_instructions.keys()}
for row in all_data:
    if row['status'] in class_counts:
        class_counts[row['status']] += 1

print("\n" + "="*45)
print("📊 CURRENT DATASET STATUS")
print("="*45)
for cat, count in class_counts.items():
    status_icon = "✅" if count >= EXAMPLES_PER_CLASS else "⏳"
    print(f"{status_icon} {cat:<22}: {count:>4} / {EXAMPLES_PER_CLASS}")
print("="*45 + "\n")

def clean_json_response(text):
    text = re.sub(r'^```json\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s*```$', '', text, flags=re.MULTILINE)
    return text.strip()

# ---------------------------------------------------------
# 4. GENERATION LOOP WITH AUTO-RETRY & CHECKPOINTS
# ---------------------------------------------------------
for category, instruction in class_instructions.items():
    existing_count = class_counts[category]

    # Skip category if we already have enough data for it from a checkpoint
    if existing_count >= EXAMPLES_PER_CLASS:
        print(f"✅ Skipping {category} (Target reached).")
        continue

    needed = EXAMPLES_PER_CLASS - existing_count
    # FIX: Use math.ceil so if we need 5 rows, it generates 1 batch of 10
    batches_needed = math.ceil(needed / BATCH_SIZE)

    print(f"\n🔄 Generating {needed} more rows for: {category} ({batches_needed} batches)...")

    for i in tqdm(range(batches_needed), desc=f"Generating {category}"):
        system_prompt = "You are an expert synthetic data generator for mental health NLP tasks. You generate realistic, first-person social media posts. Use everyday language, slang, and natural human imperfections."

        user_prompt = f"""
        Generate exactly {BATCH_SIZE} diverse, realistic text examples for the mental health category: '{category}'.
        Specific Instructions: {instruction}

        CRITICAL FORMATTING RULES:
        1. Return ONLY a valid JSON object.
        2. The JSON must have a single key named "examples".
        3. The value of "examples" must be a list of exactly {BATCH_SIZE} strings.
        """

        # --- ROBUST RETRY LOGIC FOR 429 RATE LIMITS ---
        raw_content = None
        max_retries = 2

        for attempt in range(max_retries):
            try:
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    response_format={"type": "json_object"},
                    temperature=0.9,
                    max_tokens=2000
                )
                raw_content = response.choices[0].message.content
                break  # Success! Exit the retry loop

            except RateLimitError:
                # Exponential backoff: 10s, 20s, 40s, 60s
                wait_time = min(10, 20)
                print(f"\n⚠️ 429 Rate Limit hit. Pausing for {wait_time}s to let Groq reset... (Attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
            except Exception as e:
                print(f"\n❌ API Error: {e}. Retrying in 5s...")
                time.sleep(5)

        if raw_content is None:
            print("Failed to get response after multiple retries. Skipping this batch.")
            continue

        # Parse and save
        try:
            clean_content = clean_json_response(raw_content)
            parsed_data = json.loads(clean_content)

            for text in parsed_data["examples"]:
                # Preserved your custom column name "statement"
                all_data.append({"statement": text, "status": category})

            # 🛡️ INCREMENTAL SAVE: Protects your data if you press Ctrl+C
            df_checkpoint = pd.DataFrame(all_data)
            df_checkpoint.to_csv(checkpoint_file, index=False)

            # Base sleep to prevent hitting the rate limit in the first place
            time.sleep(1.5)

        except json.JSONDecodeError:
            print("⚠️ Failed to parse JSON from model. Skipping batch.")

# ---------------------------------------------------------
# 5. AUTO-TRIM EXCESS & FINALIZE
# ---------------------------------------------------------
print("\n✂️ Trimming excess rows and finalizing dataset...")
df_final = pd.DataFrame(all_data)
trimmed_data = []

# Ensure no class has MORE than the target (due to math.ceil over-generation)
for cat in class_instructions.keys():
    cat_df = df_final[df_final['status'] == cat]
    if len(cat_df) > EXAMPLES_PER_CLASS:
        # Randomly sample exactly the target amount
        trimmed_data.append(cat_df.sample(n=EXAMPLES_PER_CLASS, random_state=42))
    else:
        trimmed_data.append(cat_df)

df_final = pd.concat(trimmed_data)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

final_filename = "/content/drive/MyDrive/Datasets/HackforHumanity/groq_synthetic_mental_health_FINAL.csv"
df_final.to_csv(final_filename, index=False)

# Clean up checkpoint file since we are done
if os.path.exists(checkpoint_file):
    os.remove(checkpoint_file)

print(f"\n🎉 SUCCESS! Final dataset contains exactly {len(df_final)} rows.")
print(f"💾 Saved final shuffled dataset to: {final_filename}")
print("\nClass Distribution:")
print(df_final['status'].value_counts())

♻️ Found existing /content/drive/MyDrive/Datasets/HackforHumanity/checkpoint_mental_health.csv. Resuming from previous progress...
Loaded 80 previously generated rows.

📊 CURRENT DATASET STATUS
⏳ Stress                :   80 / 200
⏳ Anxiety               :    0 / 200
⏳ Suicidal              :    0 / 200
⏳ Depression            :    0 / 200
⏳ Normal                :    0 / 200
⏳ Bipolar               :    0 / 200
⏳ Personality disorder  :    0 / 200


🔄 Generating 120 more rows for: Stress (12 batches)...


Generating Stress:  58%|█████▊    | 7/12 [01:03<01:03, 12.72s/it]


⚠️ 429 Rate Limit hit. Pausing for 10s to let Groq reset... (Attempt 1/2)


Generating Stress: 100%|██████████| 12/12 [01:52<00:00,  9.41s/it]



🔄 Generating 200 more rows for: Anxiety (20 batches)...


...


⚠️ 429 Rate Limit hit. Pausing for 10s to let Groq reset... (Attempt 1/2)


Generating Personality disorder:  30%|███       | 6/20 [00:54<01:50,  7.91s/it]


⚠️ 429 Rate Limit hit. Pausing for 10s to let Groq reset... (Attempt 1/2)


Generating Personality disorder: 100%|██████████| 20/20 [03:30<00:00, 10.50s/it]


✂️ Trimming excess rows and finalizing dataset...

🎉 SUCCESS! Final dataset contains exactly 1400 rows.
💾 Saved final shuffled dataset to: /content/drive/MyDrive/Datasets/HackforHumanity/groq_synthetic_mental_health_FINAL.csv

Class Distribution:
status
Depression              200
Stress                  200
Suicidal                200
Anxiety                 200
Normal                  200
Bipolar                 200
Personality disorder    200
Name: count, dtype: int64


In [ ]:
df1 = pd.read_csv('/content/drive/MyDrive/Datasets/HackforHumanity/groq_synthetic_mental_health_FINAL.csv')
df1

,statement,status
0,The only thing that keeps me from feeling wors...,Depression
1,"My phone's battery is low, but so is my energy...",Depression
2,I’m freaking out because I have a presentation...,Stress
3,"I can't see a reason to stay alive, so I keep ...",Suicidal
4,"I know I'm supposed to relax, but I can't stop...",Anxiety
...,...,...
1395,Spent $2000 on sneakers because I’m basically ...,Bipolar
1396,I just woke up at 3 a.m. and felt like I could...,Bipolar
1397,I have this insane cycle of idealizing someone...,Personality disorder
1398,"Just brewed a cup of coffee and read the news,...",Normal


In [ ]:
# 1. Load the two CSV files
df2 = pd.read_csv('/content/drive/MyDrive/Datasets/HackforHumanity/mental_health_augmented_distilbert.csv')

# 2. Combine them (stack vertically)
combined_df = pd.concat([df1, df2], ignore_index=True)

# 3. Save the result to a new CSV file
combined_df.to_csv('/content/drive/MyDrive/Datasets/HackforHumanity/combined_data.csv', index=False)

df3 = pd.read_csv("/content/drive/MyDrive/Datasets/HackforHumanity/combined_data.csv")
df3

,statement,status
0,The only thing that keeps me from feeling wors...,Depression
1,"My phone's battery is low, but so is my energy...",Depression
2,I’m freaking out because I have a presentation...,Stress
3,"I can't see a reason to stay alive, so I keep ...",Suicidal
4,"I know I'm supposed to relax, but I can't stop...",Anxiety
...,...,...
55043,"exactly, it seems like it's going to be loads ...",Normal
55044,How has your week been? Explain how your week ...,Personality disorder
55045,i shouldnt want to kill myself i have so many ...,Suicidal
55046,"Insomnia Last month, I suddenly developed the ...",Anxiety
